In [0]:
dbutils.widgets.text("catalog_name", "food_inspection")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")

catalog_name = dbutils.widgets.get("catalog_name")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

spark.sql(f"USE CATALOG {catalog_name}")

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import date

print("SCD2 Test Initialized")

In [0]:
test_name = "SCD_TEST_STORE"
test_address = "123 TEST ST"
test_zip = "60601"

print("Test constants set")

In [0]:
from datetime import date

silver_df = spark.table(f"{catalog_name}.{silver_schema}.chicago_inspections")

fake_row_1 = spark.createDataFrame(
    [(
        "TEST_SCD2",              # inspection_id
        test_name,                # dba_name
        test_name,                # aka_name
        None,                     # license_number
        "Grocery Store",          # facility_type ✅ INITIAL
        "Risk 3 (Low)",           # risk
        test_address,
        None,
        None,
        test_zip,
        date(2099,1,1),
        "TEST",
        "Pass",
        90,
        None,
        None,
        False
    )],
    schema=silver_df.schema
)

fake_row_1.write.mode("append").saveAsTable(
    f"{catalog_name}.{silver_schema}.chicago_inspections"
)

print("Inserted INITIAL record")

In [0]:
%sql
SELECT inspection_id, dba_name, facility_type
FROM food_inspection.silver.chicago_inspections
WHERE inspection_id = 'TEST_SCD2'

In [0]:
%sql
SELECT establishment_name, facility_type, is_current
FROM food_inspection.gold.dim_establishment
WHERE establishment_name = 'SCD_TEST_STORE'

In [0]:
%sql
SELECT 
    establishment_name,
    facility_type,
    is_current,
    effective_start_date,
    effective_end_date
FROM food_inspection.gold.dim_establishment
WHERE establishment_name = 'SCD_TEST_STORE'
ORDER BY effective_start_date